In [0]:
%sql
  CREATE WIDGET TEXT catalog DEFAULT 'bootcamp';
  USE CATALOG IDENTIFIER(:catalog);

In [0]:
LIST '/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/';

path,name,size,modification_time
/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,customer_changes.csv,1195,1792085040000
/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,customers.csv,52170,1792085040000
/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,order_items.csv,837571,1792085040000
/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,orders.csv,447754,1792085040000
/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,products.csv,4684,1792085040000
/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,stores.csv,943,1792085040000


# 01a – Bronze Ingestion
Loads the 6 source CSV files from the landing volume into one bronze Delta table each.
- All business columns are kept as **STRING**, so malformed values are preserved for profiling instead of becoming NULL.
- Lineage columns: `_source_file` (`_metadata.file_path`) and `_ingested_at` (`current_timestamp()`).
- Added: `_source_row_hash` (SHA-256 of all business columns) to identify exact duplicate rows during profiling.
- Tables are loaded with `INSERT OVERWRITE`, so re-running never duplicates rows (idempotent).

## Tables

In [0]:
CREATE TABLE IF NOT EXISTS bronze_efkleia_urbanmart.customers (
  customer_id       STRING COMMENT 'Raw customer_id as in file.',
  first_name        STRING COMMENT 'Raw first_name.',
  last_name         STRING COMMENT 'Raw last_name.',
  email             STRING COMMENT 'Raw email.',
  gender            STRING COMMENT 'Raw gender.',
  birth_date        STRING COMMENT 'Raw birth_date (expected yyyy-MM-dd).',
  city              STRING COMMENT 'Raw city.',
  region            STRING COMMENT 'Raw region.',
  segment           STRING COMMENT 'Raw segment.',
  signup_date       STRING COMMENT 'Raw signup_date (expected yyyy-MM-dd).',
  _source_file      STRING    NOT NULL COMMENT 'Full path of the file the row came from.',
  _source_row_hash  STRING    NOT NULL COMMENT 'SHA-256 of all raw business columns; used to detect exact duplicates.',
  _ingested_at      TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp.'
)
COMMENT 'Bronze: raw customers.csv, all STRING, full reload on each run.';

INSERT OVERWRITE bronze_efkleia_urbanmart.customers
  (customer_id, first_name, last_name, email, gender, birth_date, city, region, segment, signup_date,
   _source_file, _source_row_hash, _ingested_at)
SELECT
  customer_id, first_name, last_name, email, gender, birth_date, city, region, segment, signup_date,
  _metadata.file_path,
  sha2(to_json(struct(customer_id, first_name, last_name, email, gender, birth_date,
                      city, region, segment, signup_date)), 256),
  current_timestamp()
FROM read_files(
  '/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv',
  format => 'csv', header => true,
  schema => 'customer_id STRING, first_name STRING, last_name STRING, email STRING, gender STRING,
             birth_date STRING, city STRING, region STRING, segment STRING, signup_date STRING'
);

num_affected_rows,num_inserted_rows
508,508


In [0]:
select * from bronze_efkleia_urbanmart.customers

customer_id,first_name,last_name,email,gender,birth_date,city,region,segment,signup_date,_source_file,_source_row_hash,_ingested_at
C0001,Giorgos,Papadopoulos,giorgos.papadopoulos1@mail.gr,F,1998-12-05,Larissa,Thessaly,Retail,2022-03-31,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,19765475317c120440ff502de15523e39a5f3532c4a8192aaa9e004e81fff2a2,2026-10-17T12:04:22.886Z
C0002,Sofia,Papadopoulos,sofia.papadopoulos2@mail.gr,F,1967-11-03,Heraklion,Crete,Retail,2024-01-04,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,0fa0b7776b078dbf3e35c6989403ce003102516a0abc3c9a403a04ad0134fff9,2026-10-17T12:04:22.886Z
C0003,Petros,Makris,petros.makris3@mail.gr,F,1964-04-28,Larissa,Thessaly,Loyalty Gold,2023-12-16,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,28950e28c1853b9d7b5caf9743d1a9e9b5291e7eb55883c137ae8fe8f94e7bfd,2026-10-17T12:04:22.886Z
C0004,Dimitris,Vasileiou,dimitris.vasileiou4@mail.gr,F,1958-04-28,Patras,Western Greece,Retail,2023-01-25,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,8fe4cbc559bf4c1410e3c37cbe517afbe6bc1aa3037fa711613a872ad3e09b9d,2026-10-17T12:04:22.886Z
C0005,Dimitris,Christodoulou,dimitris.christodoulou5@mail.gr,M,1998-02-09,Athens,Attica,Loyalty Silver,2022-05-08,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,5c27208b406fadcd7afe177d897c8d801d39269f209f3456dd110cbd825aeaeb,2026-10-17T12:04:22.886Z
C0006,Giorgos,Makris,giorgos.makris6@mail.gr,M,1967-04-02,Patras,Western Greece,Retail,2023-12-23,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,02298ffa3ae9cb0f1bbc32d4d9a6a00636804c30b0cfecbcddd01145524e6157,2026-10-17T12:04:22.886Z
C0007,Maria,Ioannou,maria.ioannou7@mail.gr,F,1984-02-07,Athens,Attica,Retail,2022-10-12,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,0cd4e1ff4d39f02abae7b5bcef8685902d0de50da9df5d1274d40de8644d96cc,2026-10-17T12:04:22.886Z
C0008,Ioanna,Christodoulou,ioanna.christodoulou8@mail.gr,M,1956-05-28,Patras,Western Greece,Retail,2023-09-16,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,e5fd79dc5db0ac51b83d5eee541cfa2c939fb4037dc818799ba14066956ae6a0,2026-10-17T12:04:22.886Z
C0009,Eleni,Makris,eleni.makris9@mail.gr,F,1979-02-04,Larissa,Thessaly,Retail,2024-05-13,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,a76769fe65c1d48ca9ef2567998f586f738e350d3e5119b653df2f1d25e6ae46,2026-10-17T12:04:22.886Z
C0010,Maria,Ioannou,maria.ioannou10@mail.gr,F,1968-12-05,Ioannina,Epirus,Retail,2023-08-04,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customers.csv,a489c9d2fed6542d098ffc09fb80844fbe300ce1cf72adfb624b2158d8401c42,2026-10-17T12:04:22.886Z


In [0]:
CREATE TABLE IF NOT EXISTS bronze_efkleia_urbanmart.customer_changes (
  customer_id       STRING COMMENT 'Raw customer_id as in file.',
  new_city          STRING COMMENT 'Raw new_city.',
  new_region        STRING COMMENT 'Raw new_region.',
  new_segment       STRING COMMENT 'Raw new_segment.',
  effective_date    STRING COMMENT 'Raw effective_date (expected yyyy-MM-dd).',
  _source_file      STRING    NOT NULL COMMENT 'Full path of the file the row came from.',
  _source_row_hash  STRING    NOT NULL COMMENT 'SHA-256 of all raw business columns; used to detect exact duplicates.',
  _ingested_at      TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp.'
)
COMMENT 'Bronze: raw customer_changes.csv, all STRING, full reload on each run.';

INSERT OVERWRITE bronze_efkleia_urbanmart.customer_changes
  (customer_id, new_city, new_region, new_segment, effective_date,
   _source_file, _source_row_hash, _ingested_at)
SELECT
  customer_id, new_city, new_region, new_segment, effective_date,
  _metadata.file_path,
  sha2(to_json(struct(customer_id, new_city, new_region, new_segment, effective_date)), 256),
  current_timestamp()
FROM read_files(
  '/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv',
  format => 'csv', header => true,
  schema => 'customer_id STRING, new_city STRING, new_region STRING, new_segment STRING, effective_date STRING'
);

num_affected_rows,num_inserted_rows
25,25


In [0]:
select * from bronze_efkleia_urbanmart.customer_changes

customer_id,new_city,new_region,new_segment,effective_date,_source_file,_source_row_hash,_ingested_at
C0306,Heraklion,Crete,Retail,2025-09-18,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,3fca37a3c2f4ac7c3018546c8c265d9138cb69a3494f072c0d5419b6af022083,2026-10-17T12:04:30.332Z
C0027,Heraklion,Crete,Loyalty Gold,2025-04-12,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,e55fdb0ab8f8ad82b5db4eba9b443585bc4fd47dea218683527bb142963f6c8d,2026-10-17T12:04:30.332Z
C0397,Piraeus,Attica,Retail,2025-08-11,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,3441edecf385394639443f4ca487bcc286c74164eed57f7a38321ad5f585d8ae,2026-10-17T12:04:30.332Z
C0478,Thessaloniki,Central Macedonia,Retail,2025-06-25,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,2ef7af0a4057a0b0a070903f788c748221c5a299d9c0ce54257e0233be5163e2,2026-10-17T12:04:30.332Z
C0321,Heraklion,Crete,Loyalty Silver,2025-10-18,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,956ceb80971d42db678c1b811b36271aa35a2aec8ae30727c43d5116deb92bfa,2026-10-17T12:04:30.332Z
C0316,Ioannina,Epirus,Retail,2025-04-22,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,b96578c0b80b6d855d1c0a33d9d37351f867d5de997b2ab24220952d102cade1,2026-10-17T12:04:30.332Z
C0432,Thessaloniki,Central Macedonia,Loyalty Gold,2025-06-21,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,969134376e2ae37ea80723e94aeda0f5ace2291158e31debdb2dabc9cf3e6026,2026-10-17T12:04:30.332Z
C0339,Heraklion,Crete,Loyalty Silver,2025-06-02,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,97caf0968fc833b7540763c7ff3ab6b5ca8502ad9dddee7c0dff467975ebe0d9,2026-10-17T12:04:30.332Z
C0155,Larissa,Thessaly,Retail,2025-09-26,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,8c44950a42371eed266097662729b21296444a80b99b579275ae22d4a9dc709e,2026-10-17T12:04:30.332Z
C0252,Athens,Attica,Loyalty Gold,2025-07-18,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/customer_changes.csv,3d0ef9f678f91d7a50d070e6dcd2536c67b283b4918ffa2944c58c839e7859cc,2026-10-17T12:04:30.332Z


In [0]:
CREATE TABLE IF NOT EXISTS bronze_efkleia_urbanmart.products (
  product_id        STRING COMMENT 'Raw product_id as in file.',
  product_name      STRING COMMENT 'Raw product_name.',
  category          STRING COMMENT 'Raw category.',
  subcategory       STRING COMMENT 'Raw subcategory.',
  brand             STRING COMMENT 'Raw brand.',
  unit_cost         STRING COMMENT 'Raw unit_cost (expected decimal).',
  list_price        STRING COMMENT 'Raw list_price (expected decimal).',
  is_active         STRING COMMENT 'Raw is_active flag (expected Y/N).',
  _source_file      STRING    NOT NULL COMMENT 'Full path of the file the row came from.',
  _source_row_hash  STRING    NOT NULL COMMENT 'SHA-256 of all raw business columns; used to detect exact duplicates.',
  _ingested_at      TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp.'
)
COMMENT 'Bronze: raw products.csv, all STRING, full reload on each run.';

INSERT OVERWRITE bronze_efkleia_urbanmart.products
  (product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active,
   _source_file, _source_row_hash, _ingested_at)
SELECT
  product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active,
  _metadata.file_path,
  sha2(to_json(struct(product_id, product_name, category, subcategory, brand,
                      unit_cost, list_price, is_active)), 256),
  current_timestamp()
FROM read_files(
  '/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv',
  format => 'csv', header => true,
  schema => 'product_id STRING, product_name STRING, category STRING, subcategory STRING,
             brand STRING, unit_cost STRING, list_price STRING, is_active STRING'
);

num_affected_rows,num_inserted_rows
76,76


In [0]:
select * from bronze_efkleia_urbanmart.products

product_id,product_name,category,subcategory,brand,unit_cost,list_price,is_active,_source_file,_source_row_hash,_ingested_at
P0001,Aegean Headphones,Electronics,Headphones,Aegean,168.98,266.04,Y,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,0a665fba304b1d5a3d0816b024ad46516a73c80996faef779c898444a9682f07,2026-10-17T12:04:34.282Z
P0002,Nova Headphones,Electronics,Headphones,Nova,189.48,305.05,Y,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,6eaa0eb565b4151f4b776fac58120f28111fed0cd4124d911739ddf6b2229eae,2026-10-17T12:04:34.282Z
P0003,Helios Headphones,Electronics,Headphones,Helios,90.71,140.78,Y,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,41961f61bae8dd6b5423e1675ec498075918a5888e65094fe2bc6a2f64d28f46,2026-10-17T12:04:34.282Z
P0004,Aegean Smartwatch,Electronics,Smartwatch,Aegean,191.79,288.91,N,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,6736f9d96419895508b1461197ca4ec3ecea98f4abe75c5e46c83cc57fcda6e1,2026-10-17T12:04:34.282Z
P0005,Nova Smartwatch,Electronics,Smartwatch,Nova,205.89,251.51,Y,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,bdf69c5634ad9a795c6c7ee7e08fc5bc2299af636fcaf1a5d55191fbce62eb3b,2026-10-17T12:04:34.282Z
P0006,Helios Smartwatch,Electronics,Smartwatch,Helios,134.3,173.98,N,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,ae84c117376c4d73548a5ec4f9577f489ce6fccb843823cf61fc70e1045c5949,2026-10-17T12:04:34.282Z
P0007,Zephyr Tablet,Electronics,Tablet,Zephyr,37.73,63.16,Y,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,96629e62cc1019c214295c1cd9438b996964b86c15389ff19ebb6cb8df6b157f,2026-10-17T12:04:34.282Z
P0008,Aegean Tablet,Electronics,Tablet,Aegean,239.43,289.18,Y,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,2c55d8fc97c97fcfbb51b7bb2efb1004c41213315c0ed025b9a9fff719b5abf3,2026-10-17T12:04:34.282Z
P0009,Olympus Tablet,Electronics,Tablet,Olympus,32.56,39.19,N,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,3291ff2b0179c2708ff3ed3e4fc1d3271df3b1e0f89f59c08aa2cd0a81ef89c4,2026-10-17T12:04:34.282Z
P0010,Zephyr Speaker,Electronics,Speaker,Zephyr,54.09,92.98,Y,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/products.csv,576bbdb877b823febf1b944327e0922917d2080e1cce02ae1ee4fced06f249c8,2026-10-17T12:04:34.282Z


In [0]:
CREATE TABLE IF NOT EXISTS bronze_efkleia_urbanmart.stores (
  store_id          STRING COMMENT 'Raw store_id as in file.',
  store_name        STRING COMMENT 'Raw store_name.',
  store_type        STRING COMMENT 'Raw store_type.',
  city              STRING COMMENT 'Raw city.',
  region            STRING COMMENT 'Raw region.',
  open_date         STRING COMMENT 'Raw open_date (expected yyyy-MM-dd).',
  size_sqm          STRING COMMENT 'Raw size_sqm (expected integer).',
  _source_file      STRING    NOT NULL COMMENT 'Full path of the file the row came from.',
  _source_row_hash  STRING    NOT NULL COMMENT 'SHA-256 of all raw business columns; used to detect exact duplicates.',
  _ingested_at      TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp.'
)
COMMENT 'Bronze: raw stores.csv, all STRING, full reload on each run.';

INSERT OVERWRITE bronze_efkleia_urbanmart.stores
  (store_id, store_name, store_type, city, region, open_date, size_sqm,
   _source_file, _source_row_hash, _ingested_at)
SELECT
  store_id, store_name, store_type, city, region, open_date, size_sqm,
  _metadata.file_path,
  sha2(to_json(struct(store_id, store_name, store_type, city, region, open_date, size_sqm)), 256),
  current_timestamp()
FROM read_files(
  '/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv',
  format => 'csv', header => true,
  schema => 'store_id STRING, store_name STRING, store_type STRING, city STRING,
             region STRING, open_date STRING, size_sqm STRING'
);

num_affected_rows,num_inserted_rows
13,13


In [0]:
select * from bronze_efkleia_urbanmart.stores

store_id,store_name,store_type,city,region,open_date,size_sqm,_source_file,_source_row_hash,_ingested_at
S001,UrbanMart Athens 1,High Street,Athens,Attica,2018-06-10,1710,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,20c9ed2a1df6a6b113512c415ee42fa76ef22eb144abc50126804b749307df45,2026-10-17T12:04:38.127Z
S002,UrbanMart Piraeus 2,Outlet,Piraeus,Attica,2019-06-19,865,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,2121dad39cc53f5e01dc34962612b231fdcaa4f5e925281ca57651e5f3b472b7,2026-10-17T12:04:38.127Z
S003,UrbanMart Thessaloniki 3,Outlet,Thessaloniki,Central Macedonia,2016-05-02,1241,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,13225adc3583d5561c6963489f7e528252472667d0cf424560d5cc1376549235,2026-10-17T12:04:38.127Z
S004,UrbanMart Patras 4,High Street,Patras,Western Greece,2022-11-04,962,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,93a99112725d3fecccaab7c2fe7d1b4517bf0868e3412037ccb6514ccb5c4da1,2026-10-17T12:04:38.127Z
S005,UrbanMart Heraklion 5,Mall,Heraklion,Crete,2017-05-28,2382,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,b09e0667dc1682ccf3a7188f0629a4299a8e53026ec244d0269617092741cd2c,2026-10-17T12:04:38.127Z
S006,UrbanMart Larissa 6,High Street,Larissa,Thessaly,2018-10-18,513,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,5170d3c076e2e890a3c140aeab52175a4e3d23e175c4ed3e148cd41e620a61af,2026-10-17T12:04:38.127Z
S007,UrbanMart Ioannina 7,Mall,Ioannina,Epirus,2019-05-13,321,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,ad241ff000ae231fb52780eeb58a2448d6bffa5849f22b4989d7b226ca62243a,2026-10-17T12:04:38.127Z
S008,UrbanMart Athens 8,Mall,Athens,Attica,2016-08-18,2231,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,02f1483af2b01cb4774de9c9465e61db113790c484a9869349a89a2c37a1bed9,2026-10-17T12:04:38.127Z
S009,UrbanMart Piraeus 9,Outlet,Piraeus,Attica,2019-11-19,661,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,6d8edd880980611d6a194cd8900cc2c8251add41aae8efdefdb97e89ee097e47,2026-10-17T12:04:38.127Z
S010,UrbanMart Thessaloniki 10,Mall,Thessaloniki,Central Macedonia,2017-12-13,1208,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/stores.csv,68250712d8ad6f7ce3208727ce5f08aea03c95b8f02c7c821f471ef41687a927,2026-10-17T12:04:38.127Z


In [0]:
CREATE TABLE IF NOT EXISTS bronze_efkleia_urbanmart.orders (
  order_id          STRING COMMENT 'Raw order_id as in file.',
  order_ts          STRING COMMENT 'Raw order timestamp (expected yyyy-MM-dd HH:mm:ss; other formats possible).',
  customer_id       STRING COMMENT 'Raw customer_id (may be blank for guest orders).',
  store_id          STRING COMMENT 'Raw store_id.',
  payment_method    STRING COMMENT 'Raw payment_method.',
  order_status      STRING COMMENT 'Raw order_status.',
  _source_file      STRING    NOT NULL COMMENT 'Full path of the file the row came from.',
  _source_row_hash  STRING    NOT NULL COMMENT 'SHA-256 of all raw business columns; used to detect exact duplicates.',
  _ingested_at      TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp.'
)
COMMENT 'Bronze: raw orders.csv (order header), all STRING, full reload on each run.';

INSERT OVERWRITE bronze_efkleia_urbanmart.orders
  (order_id, order_ts, customer_id, store_id, payment_method, order_status,
   _source_file, _source_row_hash, _ingested_at)
SELECT
  order_id, order_ts, customer_id, store_id, payment_method, order_status,
  _metadata.file_path,
  sha2(to_json(struct(order_id, order_ts, customer_id, store_id, payment_method, order_status)), 256),
  current_timestamp()
FROM read_files(
  '/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv',
  format => 'csv', header => true,
  schema => 'order_id STRING, order_ts STRING, customer_id STRING, store_id STRING,
             payment_method STRING, order_status STRING'
);

num_affected_rows,num_inserted_rows
8000,8000


In [0]:
select * from bronze_efkleia_urbanmart.orders

order_id,order_ts,customer_id,store_id,payment_method,order_status,_source_file,_source_row_hash,_ingested_at
O000001,2025-09-20 08:49:00,C0147,S012,Mobile Wallet,Cancelled,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,323c40153dd0fc3b84de117f92b65349df8a7dbd7b1e3f3aa47d5373daf32a95,2026-10-17T12:04:42.209Z
O000002,2024-06-10 18:25:00,C0142,S900,Card,Completed,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,ec1493e11133852326a62c19bf22689dd0ccb3f3167f095912c4eae15658209a,2026-10-17T12:04:42.209Z
O000003,2024-10-28 12:56:00,C0060,S900,Cash,Completed,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,da3b59a87feea3ea596ec5062746c5fddb3e9ab136887a81d6afe2428e48bb13,2026-10-17T12:04:42.209Z
O000004,2024-12-27 11:18:00,C0005,S012,Cash,Completed,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,199063cf52514e5d9ee2093b174f97632117ebca7c078c666b54bcf053074147,2026-10-17T12:04:42.209Z
O000005,2024-08-11 18:59:00,C0138,S003,Mobile Wallet,Returned,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,abce1efb1415b387474d19929a1d10637548447d4e9604887b6d71a80792f61b,2026-10-17T12:04:42.209Z
O000006,2024-12-30 18:42:00,C0055,S002,Card,Returned,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,bf644d49d667bc4102ccfe4757a1ac96951d095cbd66dd8c374b30d8ac216f71,2026-10-17T12:04:42.209Z
O000007,2025-12-03 15:20:00,C0235,S002,Card,Completed,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,7307d2ee43baed5c44d9e8ac14665060ef955cc9e4db4cb76b7292c439ec4ed4,2026-10-17T12:04:42.209Z
O000008,2024-10-02 15:27:00,C0314,S006,Cash,Completed,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,06fdb6e16da52e8a58cec31736e2f4af6767adaa9634fd4c470b807089b88462,2026-10-17T12:04:42.209Z
O000009,2024-07-19 20:00:00,C0492,S012,Mobile Wallet,Completed,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,21af26728f3c8790e00246e5e4df2e31cdbc3255b344706847c27daeabdc7d46,2026-10-17T12:04:42.209Z
O000010,2024-06-16 18:06:00,C0417,S005,Cash,Completed,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/orders.csv,94c7f966ee25d96d67ad4198e1f69eaf98b42931367640209de914a877cee48f,2026-10-17T12:04:42.209Z


In [0]:
CREATE TABLE IF NOT EXISTS bronze_efkleia_urbanmart.order_items (
  order_line_id     STRING COMMENT 'Raw order_line_id as in file.',
  order_id          STRING COMMENT 'Raw order_id (FK to orders).',
  product_id        STRING COMMENT 'Raw product_id (FK to products).',
  quantity          STRING COMMENT 'Raw quantity (expected positive integer).',
  unit_price        STRING COMMENT 'Raw unit_price (expected decimal).',
  discount_pct      STRING COMMENT 'Raw discount_pct (expected fraction 0-1).',
  _source_file      STRING    NOT NULL COMMENT 'Full path of the file the row came from.',
  _source_row_hash  STRING    NOT NULL COMMENT 'SHA-256 of all raw business columns; used to detect exact duplicates.',
  _ingested_at      TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp.'
)
COMMENT 'Bronze: raw order_items.csv (order lines), all STRING, full reload on each run.';

INSERT OVERWRITE bronze_efkleia_urbanmart.order_items
  (order_line_id, order_id, product_id, quantity, unit_price, discount_pct,
   _source_file, _source_row_hash, _ingested_at)
SELECT
  order_line_id, order_id, product_id, quantity, unit_price, discount_pct,
  _metadata.file_path,
  sha2(to_json(struct(order_line_id, order_id, product_id, quantity, unit_price, discount_pct)), 256),
  current_timestamp()
FROM read_files(
  '/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv',
  format => 'csv', header => true,
  schema => 'order_line_id STRING, order_id STRING, product_id STRING,
             quantity STRING, unit_price STRING, discount_pct STRING'
);

num_affected_rows,num_inserted_rows
24111,24111


In [0]:
select * from bronze_efkleia_urbanmart.order_items

order_line_id,order_id,product_id,quantity,unit_price,discount_pct,_source_file,_source_row_hash,_ingested_at
L0000001,O000001,P0002,2,305.05,0.1,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,0547edddd7962907ae0d56240df8cd15d093a9f856918dadde97d6dfda7ca00f,2026-10-17T12:04:46.424Z
L0000002,O000001,P0037,2,100.15,0.1,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,95b727c5ad1651c9b14c59a68764a4ebf0a9de6b05c560c4ab1a8120aaa5c8ba,2026-10-17T12:04:46.424Z
L0000003,O000001,P0066,2,156.8,0.2,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,82b6680a11a65d134605cfd45c40821ab607e2c899f33241bd85a4c772730272,2026-10-17T12:04:46.424Z
L0000004,O000001,P0005,3,251.51,0,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,cab2bc3d515381499549c14714271590a5b7246e6a85fd33ac4750fcb49079e4,2026-10-17T12:04:46.424Z
L0000005,O000002,P0002,1,305.05,0.1,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,93ff076f8c3c678b9b973399724903e23c4650d7ba6dac030005d3a685adbeac,2026-10-17T12:04:46.424Z
L0000006,O000002,P0037,3,100.15,0,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,7480f33ffd85be5e392513b902b984ec72d8bf1c1f1c6f14c523c3fe1b9b2478,2026-10-17T12:04:46.424Z
L0000007,O000002,P0029,2,260.02,0.05,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,714ce80ae9513115dc06c478beb0ac76d01df1600f36881e83be3b6249ea751d,2026-10-17T12:04:46.424Z
L0000008,O000002,P0074,1,182.7,0.05,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,08c8b579360e031ebd0012b2a5fbeadf9e775d3b035c381ffea2b0ce966b5b42,2026-10-17T12:04:46.424Z
L0000009,O000002,P0065,1,103.78,0.1,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,f15c096cf05a9e0838524decb231aec8b3c752cc408695fddd92ec5d48190ed3,2026-10-17T12:04:46.424Z
L0000010,O000003,P0049,4,74.88,0,dbfs:/Volumes/bootcamp/bronze_efkleia_urbanmart/landing/order_items.csv,68d287386b845a1fa4507e818be8c6327919c9350106d7adfbb5ce658691140e,2026-10-17T12:04:46.424Z
